# M5.S3 Modules and Apptainer

We will select cluster software with modules, compile a small C program, then run a PyTorch program inside an existing Apptainer image.

For each step, read why we do it, then type the commands in SSH or run the matching notebook cell. Run All follows the same sequence. The commands request CPU resources through Slurm, so they may wait for an allocation. No GPU or software installation is needed.

## 1. Open your working folder

Connect from your computer and create a folder for the exercise:

```bash
ssh YOUR_USERNAME@ssh.iesci.tech
mkdir -p ~/hpc-labs/m5s3-basics
cd ~/hpc-labs/m5s3-basics
```

The next cell selects that same folder in Jupyter. A `%%writefile` cell saves the code below it. In SSH, open the named file with `nano` and paste the code, leaving out the `%%writefile` line.

In [ ]:
from pathlib import Path
import os

folder = Path.home() / 'hpc-labs' / 'm5s3-basics'
folder.mkdir(parents=True, exist_ok=True)
os.chdir(folder)
print('Working folder:', folder)

The small helper below runs shell commands and displays their output. It keeps the notebook's Slurm allocation separate from the resources requested by each exercise. You do not need this helper in SSH.

In [ ]:
import subprocess
import selectors

def run(command):
    print('$ ' + command, flush=True)
    # Jupyter can itself be inside a small Slurm job. Each exercise requests
    # its own allocation; only the child shell loses inherited job/step limits.
    keep = {'SLURM_CONF', 'SLURM_CONF_SERVER', 'SLURM_CLUSTERS',
            'SLURM_ACCOUNT', 'SLURM_QOS'}
    environment = {key: value for key, value in os.environ.items()
                   if not key.startswith('SLURM_') or key in keep}
    process = subprocess.Popen(['bash', '-lc', 'set -euo pipefail\n' + command],
                               env=environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    # Print Slurm/compiler output as it arrives, including while a job queues.
    reader = selectors.DefaultSelector()
    reader.register(process.stdout, selectors.EVENT_READ)
    while process.poll() is None:
        if reader.select(timeout=15):
            line = process.stdout.readline()
            if line: print(line, end='', flush=True)
        else:
            print('Still waiting for the command. Slurm resources may be queued.', flush=True)
    for line in process.stdout: print(line, end='', flush=True)
    reader.close()
    if process.wait() != 0:
        print('Command failed. Inspect the output above.', flush=True)
        raise RuntimeError('The command did not complete successfully.')

## 2. Look at the software available

`module list` shows the modules currently selected. `module spider` describes a named module and any prerequisites. We ask for specific versions to keep the output short.

In SSH:

```bash
hostname
module list
module spider foss/2023b
module spider HDF5/1.14.3-gompi-2023b
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
hostname
module list
module spider foss/2023b
module spider HDF5/1.14.3-gompi-2023b
""")

Write down one compiler/MPI stack and one scientific-library module. A module selects installed software; it does not install a package in your account.

## 3. Select a compiler and build a program

`module purge` clears the optional module selection in this shell. `foss/2023b` loads a coherent scientific software stack. These exact versions were present in the supplied SciTech run.

The C program prints the compiler used and adds the integers from 1 to 100,000. The known answer lets us check the run.

In [ ]:
%%writefile module_demo.c
#include <stdio.h>

int main(void) {
    /* Use a wide integer because the sum is larger than a 32-bit signed int. */
    long long total = 0;

    for (int number = 1; number <= 100000; number++) {
        total += number;
    }

    /* __VERSION__ identifies the compiler that built this executable. */
    printf("Compiler: %s\n", __VERSION__);
    printf("Sum 1..100000: %lld\n", total);
    return 0;
}

In SSH:

```bash
module purge
module load foss/2023b
module list
which gcc
gcc --version | head -1
gcc -O2 -Wall module_demo.c -o module_demo
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
module purge
module load foss/2023b
module list
which gcc
gcc --version | head -1
gcc -O2 -Wall module_demo.c -o module_demo
""")

Compilation creates `module_demo`. Now request one CPU core and run it. `tee` displays the output and saves it in a file.

In SSH:

```bash
module load foss/2023b
srun -p cpu -N1 -n1 -c1 --mem=1G -t 00:02:00 ./module_demo | tee module_result.txt
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
module load foss/2023b
srun -p cpu -N1 -n1 -c1 --mem=1G -t 00:02:00 ./module_demo | tee module_result.txt
""")

Check that the sum is `5000050000`. The compiler should be the one selected by the module. Why does the C program use `long long`?

## 4. Save and restore the module selection

Save this selection under a name, clear it, then restore it. The two compiler paths should match. The C executable and output file remain in your folder when modules are unloaded.

In SSH:

```bash
module purge
module load foss/2023b
which gcc > compiler-before.txt
module save m5s3_class
module purge
module list
module restore m5s3_class
which gcc > compiler-after.txt
cmp compiler-before.txt compiler-after.txt
cat module_result.txt
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
module purge
module load foss/2023b
which gcc > compiler-before.txt
module save m5s3_class
module purge
module list
module restore m5s3_class
which gcc > compiler-after.txt
cmp compiler-before.txt compiler-after.txt
cat module_result.txt
""")

An empty `cmp` output means the paths match. Explain why a saved module selection would stop being usable if an administrator removed one of its module versions.

## 5. Load a scientific library

HDF5 is used to store scientific data. Its compiler wrapper supplies paths for programs that use the library. Here we only inspect the installation.

In SSH:

```bash
module purge
module load foss/2023b
module load HDF5/1.14.3-gompi-2023b
module list
which h5pcc
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
module purge
module load foss/2023b
module load HDF5/1.14.3-gompi-2023b
module list
which h5pcc
""")

The supplied cluster run found `h5pcc` in the HDF5 installation. Explain what selecting a library module gives you that merely copying your C source does not.

## 6. Inspect the existing Apptainer image

The image below already exists on SciTech. `apptainer --version` identifies the host runtime; the Python command identifies the interpreter and PyTorch inside the image. The notebook interpreter does not need PyTorch installed.

`--cleanenv` reduces inherited host environment settings. This example uses CPUs, even though the image filename mentions CUDA.

In SSH:

```bash
apptainer --version
ls -lh /data/software/containers/pytorch-2.8.0-cuda12.8.sif
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 -c 'import torch; print(torch.__version__)'
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
apptainer --version
ls -lh /data/software/containers/pytorch-2.8.0-cuda12.8.sif
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 -c 'import torch; print(torch.__version__)'
""")

The supplied run reported Apptainer 1.5.4 and PyTorch 2.8.0+cu128. Your own command output is the record of what you used. A container does not reserve CPU cores; that is what `srun` does.

## 7. Run the same program twice

The model learns a line close to `y = 3x + 2`. Read the training loop: clear gradients, make a prediction, calculate the loss and gradients, then update the parameters. Its assertions check the learned slope, intercept and error.

In [ ]:
%%writefile train_model.py
"""Fit a straight line inside the existing PyTorch container."""
import json
import sys
import torch

# Read the two experiment parameters from the command line.
seed = int(sys.argv[1])
true_slope = float(sys.argv[2])

# Fix the random initialisation and keep the CPU workload within our allocation.
torch.manual_seed(seed)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)

# Generate the same 256 observations whenever the slope parameter is unchanged.
x = torch.linspace(-2, 2, 256).reshape(-1, 1)
y = true_slope * x + 2.0 + 0.05 * torch.sin(9 * x)

# The model learns one slope and one intercept.
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(240):
    optimizer.zero_grad()                 # Discard the previous gradients.
    prediction = model(x)
    loss = torch.mean((prediction - y) ** 2)
    loss.backward()                      # Calculate the new gradients.
    optimizer.step()                     # Update the slope and intercept.

# Inspect the learned values without creating more gradient calculations.
with torch.no_grad():
    slope = model.weight.item()
    intercept = model.bias.item()
    error = torch.mean(torch.abs(model(x) - y)).item()

# A program finishing is not enough: check that it learned the intended line.
assert abs(slope - true_slope) < 0.1
assert abs(intercept - 2.0) < 0.1
assert error < 0.1

# Stable, sorted output lets us compare separate executions with cmp.
result = {
    "torch_version": torch.__version__,
    "seed": seed,
    "target_slope": true_slope,
    "learned_slope": round(slope, 6),
    "learned_intercept": round(intercept, 6),
    "mean_absolute_error": round(error, 6),
}
print(json.dumps(result, sort_keys=True))

Run A uses seed 42 and target slope 3. The shell sends the script through standard input with `<`, so this run does not need a bind mount. The JSON output is saved on the host by `tee`.

In SSH:

```bash
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 - 42 3.0 < train_model.py | tee run_A.json
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 - 42 3.0 < train_model.py | tee run_A.json
""")

Start a separate container process with the same image, code, seed and inputs. Compare the result with A.

In SSH:

```bash
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 - 42 3.0 < train_model.py | tee run_B.json
cmp run_A.json run_B.json
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 - 42 3.0 < train_model.py | tee run_B.json
cmp run_A.json run_B.json
""")

`cmp` should print nothing. The learned slope should be near 3, the intercept near 2 and the mean absolute error below 0.1. This checks repeated execution with these settings; it does not promise identical floating-point results on all hardware or software versions.

## 8. Change one input

Change only the target slope from 3 to 4. Keep the image, program and seed fixed. The learned slope should move near 4 while the intercept remains near 2.

In SSH:

```bash
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 - 42 4.0 < train_model.py | tee run_C.json
cat run_C.json
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
srun -p cpu -N1 -n1 -c2 --mem=2G -t 00:03:00 \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 - 42 4.0 < train_model.py | tee run_C.json
cat run_C.json
""")

Explain why changing the input can change the result even though the container and code are unchanged.

## 9. Share a folder with the container

A bind mount gives a host folder a path inside the container. `--bind "$PWD:/work"` makes our current folder available as `/work`; `--pwd /work` starts the program there.

This small program shows the folder, checks that our training script is visible and writes a note. We will read that note from the host after the process finishes.

In [ ]:
%%writefile write_note.py
from pathlib import Path

# /work is the host folder selected by the bind option.
print("Container folder:", Path.cwd())
print("Training program visible:", Path("train_model.py").is_file())

# The host will keep this file after the container process ends.
Path("from_container.txt").write_text("Hello from inside the container.\n")

In SSH:

```bash
srun -p cpu -N1 -n1 -c1 --mem=1G -t 00:02:00 \
    apptainer exec --cleanenv --bind "$PWD:/work" --pwd /work \
    /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 write_note.py
cat from_container.txt
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
srun -p cpu -N1 -n1 -c1 --mem=1G -t 00:02:00 \
    apptainer exec --cleanenv --bind "$PWD:/work" --pwd /work \
    /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 write_note.py
cat from_container.txt
""")

Expect the container folder to be `/work`, the training script to be visible, and the host to read `Hello from inside the container.` The output stays in your folder because `/work` maps to that folder. It is not written into the SIF image.

## 10. Start containers on two servers with Slurm

Now ask Slurm for two CPU nodes and start one container process on each. Each process prints the server name from inside its container.

`-N2` requests two nodes. `-n2 --ntasks-per-node=1` places one task on each node. `-c1` gives each task one CPU core. `--label` adds the task number to its output.

Run in SSH or execute the next cell:

```bash
srun -p cpu -N2 -n2 --ntasks-per-node=1 -c1 --mem=1G -t 00:02:00 --label \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 -c 'import socket; print("Container on", socket.gethostname(), flush=True)' \
    | tee two_nodes.txt
```

Expect two labelled lines containing two different server names. Both processes start under the same Slurm allocation. Slurm obtains the resources and launches the tasks; Apptainer supplies the software environment on each server.

These are independent processes. They do not communicate or train one shared model. The command may wait until two nodes are available, and the shared image must be readable on both.

In [ ]:
run(r"""
srun -p cpu -N2 -n2 --ntasks-per-node=1 -c1 --mem=1G -t 00:02:00 --label \
    apptainer exec --cleanenv /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /opt/conda/bin/python3 -c 'import socket; print("Container on", socket.gethostname(), flush=True)' \
    | tee two_nodes.txt
""")

## 11. Read your results

The final cell checks the saved outputs and shows the learned values. These are results from your runs, not example values.

In [ ]:
import json

native = Path('module_result.txt').read_text()
assert 'Sum 1..100000: 5000050000' in native
assert Path('compiler-before.txt').read_text() == Path('compiler-after.txt').read_text()

results = [json.loads(Path('run_' + name + '.json').read_text()) for name in ['A', 'B', 'C']]
assert results[0] == results[1]
assert results[0] != results[2]
for result in results:
    assert abs(result['learned_slope'] - result['target_slope']) < 0.1
    assert abs(result['learned_intercept'] - 2.0) < 0.1
    assert result['mean_absolute_error'] < 0.1
assert Path('from_container.txt').read_text() == 'Hello from inside the container.\n'

servers = {line.split()[-1] for line in Path('two_nodes.txt').read_text().splitlines()
           if 'Container on' in line}
assert len(servers) == 2

print('C result: PASS')
print('Module restore: PASS')
print('Repeated container runs A/B: PASS')
print('Changed input C: PASS')
print('Container file visible on host: PASS')
print('Containers on two servers: PASS - ' + ', '.join(sorted(servers)))
print('\nRun   Target slope   Learned slope   Intercept   Mean absolute error')
for name, result in zip(['A', 'B', 'C'], results):
    print(f'{name:5s} {result["target_slope"]:12.3f} {result["learned_slope"]:15.6f} '
          f'{result["learned_intercept"]:11.6f} {result["mean_absolute_error"]:21.6f}')

## Your next experiment

Change the seed from 42 to 7 and repeat A and B. Predict whether the initial model will change and whether it should still learn the intended line. Then change the number of epochs, keeping the other settings fixed.

Record the selected modules, compiler version, image path, PyTorch version, seed, program and results. For file identity, you can also run `sha256sum train_model.py run_A.json run_B.json`. A filename alone does not identify file contents.

Explain the difference between selecting host software with modules, packaging software in an image, and using Slurm to obtain resources. The [extended notebook](M5S3_containers_portability_reproducibility_extended.ipynb) has the longer dependency, cache and handover exercises.

## 12. Let the two containers communicate with MPI

The previous command launched two independent processes. This program connects two ranks with MPI. Rank 0 contributes 1, rank 1 contributes 2, and `MPI_Allreduce` returns 3 to both.

We compile with the host's MPI stack. The container run below exposes that same stack, rather than assuming the PyTorch image contains a compatible MPI. Read the rank/size check and the Allreduce call first.

In [ ]:
%%writefile mpi_demo.c
#include <mpi.h>
#include <stdio.h>

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank;
    int ranks;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &ranks);

    /* Require one communicator containing both processes. */
    if (ranks != 2) {
        MPI_Abort(MPI_COMM_WORLD, 1);
    }

    char server[MPI_MAX_PROCESSOR_NAME];
    int name_length;
    MPI_Get_processor_name(server, &name_length);

    /* Rank 0 contributes 1 and rank 1 contributes 2. Both receive the sum. */
    int contribution = rank + 1;
    int total;
    MPI_Allreduce(&contribution, &total, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
    if (total != 3) {
        MPI_Abort(MPI_COMM_WORLD, 1);
    }

    printf("MPI rank %d/%d on %s: sum = %d\n", rank, ranks, server, total);
    if (rank == 0) {
        printf("MPI_SUM_PASS: 3\n");
    }

    MPI_Finalize();
    return 0;
}

In SSH:

```bash
module purge
module load foss/2023b
mpicc -O2 -Wall mpi_demo.c -o mpi_demo
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
module purge
module load foss/2023b
mpicc -O2 -Wall mpi_demo.c -o mpi_demo
""")

### Allocate two servers and launch one MPI rank on each

`salloc` reserves the two nodes for this command and releases them when it finishes. The host `mpirun` starts one container per node. `MPI_ROOT` comes from the selected compiler path. Binding `/cvmfs` exposes the site's MPI installation and its dependencies; `LD_LIBRARY_PATH` and `OPAL_PREFIX` select that MPI inside the container.

This small example uses ordinary TCP messaging. We preserve the MPI launcher's environment here, so this command intentionally does not use `--cleanenv`. The work folder is still mounted as `/work`.

This is the host-MPI bind approach described in the [Apptainer MPI guide](https://apptainer.org/docs/user/latest/mpi.html). It depends on the site's MPI/runtime integration, which this new example still needs to verify on SciTech.

In SSH:

```bash
module purge
module load foss/2023b
MPI_ROOT=$(dirname "$(dirname "$(command -v mpicc)")")
export OMPI_MCA_pml=ob1
export OMPI_MCA_btl=self,tcp
salloc -p cpu -N2 -n2 --ntasks-per-node=1 -c1 --mem=1G -t 00:03:00 \
    mpirun -np 2 --map-by ppr:1:node \
    apptainer exec --bind "$PWD:/work" --bind /cvmfs:/cvmfs:ro --pwd /work \
    --env LD_LIBRARY_PATH="$MPI_ROOT/lib:${LD_LIBRARY_PATH:-}" \
    --env OPAL_PREFIX="$MPI_ROOT" \
    /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /work/mpi_demo | tee mpi_two_nodes.txt
```

Run the same commands in Jupyter:

In [ ]:
run(r"""
module purge
module load foss/2023b
MPI_ROOT=$(dirname "$(dirname "$(command -v mpicc)")")
export OMPI_MCA_pml=ob1
export OMPI_MCA_btl=self,tcp
salloc -p cpu -N2 -n2 --ntasks-per-node=1 -c1 --mem=1G -t 00:03:00 \
    mpirun -np 2 --map-by ppr:1:node \
    apptainer exec --bind "$PWD:/work" --bind /cvmfs:/cvmfs:ro --pwd /work \
    --env LD_LIBRARY_PATH="$MPI_ROOT/lib:${LD_LIBRARY_PATH:-}" \
    --env OPAL_PREFIX="$MPI_ROOT" \
    /data/software/containers/pytorch-2.8.0-cuda12.8.sif \
    /work/mpi_demo | tee mpi_two_nodes.txt
""")

Expect rank `0/2` and rank `1/2` on different server names, each with `sum = 3`, followed by `MPI_SUM_PASS: 3`. Two separate `rank 0/1` processes are not a successful MPI run; the program rejects that case.

Explain the roles of Slurm, `mpirun`, Apptainer and `MPI_Allreduce`. What extra check does this provide compared with simply printing two hostnames?

In [ ]:
mpi_log = Path('mpi_two_nodes.txt').read_text()
assert 'MPI_SUM_PASS: 3' in mpi_log
assert 'MPI rank 0/2' in mpi_log and 'MPI rank 1/2' in mpi_log
mpi_servers = {line.split(' on ', 1)[1].split(':', 1)[0]
               for line in mpi_log.splitlines() if 'MPI rank ' in line}
assert len(mpi_servers) == 2
print('MPI + Apptainer on two servers: PASS (1 + 2 = 3)')